# Muscle Fiber Workflow

Load the synthetic `rectus_femoris` dataset, inspect its fiber-direction vector field, visualize it in three complementary ways, and export line glyphs for ParaView/OpenSim.

In [ ]:
import numpy as np

from pyfieldml import datasets

doc = datasets.load_rectus_femoris()
print("Region:", doc.region.name)

## Inspect the fiber field

In [ ]:
fibers = doc.evaluators["fiber_direction"]
fv = fibers.as_ndarray()
print("fiber vectors shape:", fv.shape)
print("norm check:", np.allclose(np.linalg.norm(fv, axis=1), 1.0, atol=1e-6))
print("dominant axis (argmax of |component|):")
print("  axis count:", np.bincount(np.argmax(np.abs(fv), axis=1), minlength=3))

## Three-panel pyvista view: wireframe, solid+glyphs, streamlines

Left: bare topology. Middle: surface + fiber glyphs colored by magnitude. Right: streamlines seeded inside the muscle belly to trace the fiber field through the volume.

In [ ]:
import pyvista as pv

from pyfieldml.interop.pyvista import to_pyvista

pv.OFF_SCREEN = True
pv.set_jupyter_backend("static")

grid = to_pyvista(doc)
coords = doc.evaluators["coordinates"].as_ndarray()
grid.point_data["fiber"] = fv
grid.point_data["|fiber|"] = np.linalg.norm(fv, axis=1)

# Short line glyphs at every node
centers = coords
bounds_arr = np.array(grid.bounds)
bounds_diag = float(np.linalg.norm(bounds_arr[1::2] - bounds_arr[0::2]))
scale = 0.03 * bounds_diag
glyph_grid = pv.PolyData(centers)
glyph_grid["fiber"] = fv
glyphs = glyph_grid.glyph(orient="fiber", scale=False, factor=scale, geom=pv.Line())

p = pv.Plotter(off_screen=True, window_size=(1100, 450), shape=(1, 3))
p.subplot(0, 0)
p.add_mesh(grid, style="wireframe", color="steelblue", line_width=1)
p.add_text("wireframe", font_size=10)
p.view_isometric()
p.subplot(0, 1)
p.add_mesh(grid, color="lightsalmon", opacity=0.35, show_edges=False)
p.add_mesh(glyphs, color="crimson")
p.add_text("fiber glyphs", font_size=10)
p.view_isometric()
p.subplot(0, 2)
try:
    streams = grid.streamlines_from_source(
        pv.PolyData(centers[:: max(1, len(centers) // 30)]),
        vectors="fiber",
        max_time=5.0,
        initial_step_length=0.05,
    )
    p.add_mesh(grid, color="lightsalmon", opacity=0.2)
    p.add_mesh(streams.tube(radius=scale * 0.05), color="crimson")
    p.add_text("streamlines", font_size=10)
except Exception as exc:
    p.add_mesh(grid, color="lightsalmon", opacity=0.35)
    p.add_mesh(glyphs, color="crimson")
    p.add_text(f"streamlines failed: {type(exc).__name__}", font_size=9)
p.view_isometric()
p.show(screenshot="/tmp/muscle_triple.png")

## Alignment histogram

The rectus femoris runs proximo-distally, so most fibers should have a strong z-component. Plotting the distribution confirms the synthetic field is anatomically plausible.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(10, 3), sharey=True)
for ax, comp, label in zip(axes, fv.T, ["x", "y", "z"], strict=True):
    ax.hist(comp, bins=24, color="steelblue", edgecolor="black")
    ax.set_xlabel(f"fiber_{label}")
    ax.set_xlim(-1.05, 1.05)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("node count")
fig.suptitle("rectus_femoris fiber-vector component distributions")
fig.tight_layout()

## Export fiber glyphs as VTK

`export_fiber_lines` writes a short line segment at each node along the fiber direction. Open the result in ParaView to see the fiber arrangement.

In [ ]:
import tempfile
from pathlib import Path

from pyfieldml.interop.opensim import export_fiber_lines

out_dir = Path(tempfile.mkdtemp(prefix="rectus_"))
out = export_fiber_lines(
    doc,
    field="fiber_direction",
    out_path=out_dir / "fibers.vtu",
    length_scale=0.01,
)
print("wrote", out, f"({out.stat().st_size} bytes)")

### Licensing note

The bundled `rectus_femoris` dataset is a **synthetic** spindle shape authored for demonstration (CC0). Real Physiome-project muscle meshes require separate licensing; see the `datasets.info('rectus_femoris')` metadata.